# Workshop 2: Exploratory Data Profiling & Quality Risk Analysis
**Course:** ETL - Data Engineering and Artificial Intelligence (UAO)  
**Authors:** Juan David & AI Senior Data Engineering Assistant  
**Objective:** Profile both heterogeneous data sources (`spotify_dataset.csv` and `the_grammy_awards.csv`), analyze cross-source consistency, and gather empirical evidence to justify quality rules, metric thresholds, and dimensional design decisions.

---
## Analytical Scope Alignment
This profiling specifically investigates attributes supporting our three mandatory analytical requirements:
* **REQ-01:** Temporal Evolution of Historical Winners' Commercial Success (`the_grammy_awards.year`, `nominee`, `winner`, `spotify_dataset.track_name`, `popularity`).
* **REQ-02:** Acoustic Behavior in Pop and Urban Genres (`the_grammy_awards.artist`, `winner`, `spotify_dataset.artists`, `track_genre`, `speechiness`, `energy`).
* **REQ-03:** Genre Diversity in Multi-Awarded Artists (`the_grammy_awards.artist`, `category`, `winner`, `spotify_dataset.artists`, `track_genre`).


## 1. Environment and Data Ingestion

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set plot styles
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

DATA_DIR = Path("../data")
spotify_path = DATA_DIR / "spotify_dataset.csv"
grammys_path = DATA_DIR / "the_grammy_awards.csv"

df_spotify = pd.read_csv(spotify_path)
df_grammys = pd.read_csv(grammys_path)

print(f"Spotify Dataset Shape: {df_spotify.shape}")
print(f"Grammy Awards Shape:   {df_grammys.shape}")


Spotify Dataset Shape: (114000, 21)
Grammy Awards Shape:   (4810, 10)


## 2. Structural Analysis (Schema & Data Types)
We examine the declared types and column definitions against the requirements.


In [2]:
print("=== SPOTIFY DATA TYPES ===")
print(df_spotify.dtypes)
print("\n=== GRAMMYS DATA TYPES ===")
print(df_grammys.dtypes)


=== SPOTIFY DATA TYPES ===
Unnamed: 0            int64
track_id                str
artists                 str
album_name              str
track_name              str
popularity            int64
duration_ms           int64
explicit               bool
danceability        float64
energy              float64
key                   int64
loudness            float64
mode                  int64
speechiness         float64
acousticness        float64
instrumentalness    float64
liveness            float64
valence             float64
tempo               float64
time_signature        int64
track_genre             str
dtype: object

=== GRAMMYS DATA TYPES ===
year            int64
title             str
published_at      str
updated_at        str
category          str
nominee           str
artist            str
workers           str
img               str
winner           bool
dtype: object


## 3. Completeness Analysis (Missing Values)
We evaluate missing-value counts and percentages for both sources.


In [3]:
def completeness_report(df, name):
    null_counts = df.isnull().sum()
    null_pct = (null_counts / len(df)) * 100
    report = pd.DataFrame({
        "Missing_Count": null_counts,
        "Missing_Percentage": null_pct.round(4)
    })
    return report[report["Missing_Count"] > 0]

print("=== SPOTIFY MISSING VALUES ===")
spot_missing = completeness_report(df_spotify, "Spotify")
print(spot_missing)

print("\n=== GRAMMY AWARDS MISSING VALUES ===")
gram_missing = completeness_report(df_grammys, "Grammys")
print(gram_missing)


=== SPOTIFY MISSING VALUES ===
            Missing_Count  Missing_Percentage
artists                 1              0.0009
album_name              1              0.0009
track_name              1              0.0009

=== GRAMMY AWARDS MISSING VALUES ===
         Missing_Count  Missing_Percentage
nominee              6              0.1247
artist            1840             38.2536
workers           2190             45.5301
img               1367             28.4200


## 4. Uniqueness & Candidate Keys Analysis
We test whether candidate keys (`track_id` in Spotify, `(year, category, nominee)` in Grammys) are unique.


In [4]:
print(f"Spotify Total Rows: {len(df_spotify)}")
print(f"Spotify Unique track_id: {df_spotify['track_id'].nunique()}")
print(f"Spotify Duplicate track_id count: {df_spotify['track_id'].duplicated().sum()}")

# Check why track_id duplicates exist
dup_samples = df_spotify[df_spotify['track_id'].duplicated(keep=False)].sort_values('track_id').head(6)
print("\nSample of track_id appearing across multiple genres:")
print(dup_samples[['track_id', 'track_name', 'artists', 'track_genre']])

print("\nTesting (track_id, track_genre) as composite candidate key:")
composite_dups = df_spotify.duplicated(subset=['track_id', 'track_genre']).sum()
print(f"Duplicates at (track_id, track_genre) grain: {composite_dups}")


Spotify Total Rows: 114000
Spotify Unique track_id: 89741
Spotify Duplicate track_id count: 24259

Sample of track_id appearing across multiple genres:
                      track_id                                track_name  \
15028   001APMDOl3qtx1526T11n1                                    Better   
103211  001APMDOl3qtx1526T11n1                                    Better   
85578   001YQlnDSduXd5LgBd66gT  El Tiempo Es Dinero - Remasterizado 2007   
100420  001YQlnDSduXd5LgBd66gT  El Tiempo Es Dinero - Remasterizado 2007   
2106    003vvx7Niy0yvhvHt4a68B                            Mr. Brightside   
3257    003vvx7Niy0yvhvHt4a68B                            Mr. Brightside   

                  artists  track_genre  
15028   Pink Sweat$;Kirby        chill  
103211  Pink Sweat$;Kirby         soul  
85578         Soda Stereo    punk-rock  
100420        Soda Stereo          ska  
2106          The Killers     alt-rock  
3257          The Killers  alternative  

Testing (track_id, track_ge

## 5. Numerical Measures & Range Boundaries
Verifying acoustic metrics and popularity ranges for REQ-01 and REQ-02.


In [5]:
num_cols = ['popularity', 'energy', 'speechiness', 'danceability', 'acousticness', 'tempo']
print("=== SPOTIFY NUMERICAL METRICS SUMMARY ===")
print(df_spotify[num_cols].describe().round(4))


=== SPOTIFY NUMERICAL METRICS SUMMARY ===


        popularity       energy  speechiness  danceability  acousticness  \
count  114000.0000  114000.0000  114000.0000   114000.0000   114000.0000   
mean       33.2385       0.6414       0.0847        0.5668        0.3149   
std        22.3051       0.2515       0.1057        0.1735        0.3325   
min         0.0000       0.0000       0.0000        0.0000        0.0000   
25%        17.0000       0.4720       0.0359        0.4560        0.0169   
50%        35.0000       0.6850       0.0489        0.5800        0.1690   
75%        50.0000       0.8540       0.0845        0.6950        0.5980   
max       100.0000       1.0000       0.9650        0.9850        0.9960   

             tempo  
count  114000.0000  
mean      122.1478  
std        29.9782  
min         0.0000  
25%        99.2188  
50%       122.0170  
75%       140.0710  
max       243.3720  


## 6. Temporal Content & Award History
Checking ceremony year bounds and distribution for REQ-01.


In [6]:
print("=== GRAMMY YEAR BOUNDS ===")
print(f"Min Year: {df_grammys['year'].min()} | Max Year: {df_grammys['year'].max()}")

# Decade grouping
df_grammys['decade'] = (df_grammys['year'] // 10) * 10
decade_dist = df_grammys.groupby('decade').size()
print("\nAwards count by decade:")
print(decade_dist)


=== GRAMMY YEAR BOUNDS ===
Min Year: 1958 | Max Year: 2019

Awards count by decade:
decade
1950      63
1960     433
1970     487
1980     684
1990     866
2000    1067
2010    1210
dtype: int64


## 7. Cross-Source Consistency & Match Analysis
Evaluating the empirical feasibility of joining Spotify with Grammy Awards.


In [7]:
# Artist matching
spot_artists = set(df_spotify['artists'].dropna().str.lower().str.strip())
gram_artists = set(df_grammys['artist'].dropna().str.lower().str.strip())

direct_artist_matches = spot_artists.intersection(gram_artists)
print(f"Distinct Spotify Artists: {len(spot_artists)}")
print(f"Distinct Grammy Artists:  {len(gram_artists)}")
print(f"Direct Artist String Matches: {len(direct_artist_matches)}")

# Track Name vs Nominee matching
spot_tracks = set(df_spotify['track_name'].dropna().str.lower().str.strip())
gram_nominees = set(df_grammys['nominee'].dropna().str.lower().str.strip())

direct_track_matches = spot_tracks.intersection(gram_nominees)
print(f"Direct Song Title / Nominee Matches: {len(direct_track_matches)}")


Distinct Spotify Artists: 31428


Distinct Grammy Artists:  1656
Direct Artist String Matches: 408
Direct Song Title / Nominee Matches: 706


## 8. Data Quality Risk & Rules Matrix (PDF Section 6.3 & 6.5)

| Rule ID | Dataset / Layer | Attribute(s) | Quality Dimension | Quality Rule Statement | Metric / Threshold | Severity | Related Requirement |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **DQ-RAW-01** | Spotify Raw | `track_id` | Completeness & Uniqueness | Track identifier must be non-null | mostly = 1.0 (100%) | **Critical** | All (Grain integrity) |
| **DQ-RAW-02** | Spotify Raw | `energy`, `speechiness` | Validity & Range | Acoustic features must be normalized ratios between 0.0 and 1.0 | 100% within [0.0, 1.0] | **Critical** | REQ-02 (Acoustic gap) |
| **DQ-RAW-03** | Spotify Raw | `artists` | Completeness | Artist string must not be missing | mostly = 0.999 (99.9%) | **Warning** | REQ-02, REQ-03 |
| **DQ-RAW-04** | Grammys Raw | `year` | Validity & Range | Ceremony year must be between 1958 and current year | 100% within [1958, 2026] | **Critical** | REQ-01 (Temporal trends) |
| **DQ-RAW-05** | Grammys Raw | `winner` | Validity & Consistency | Winner flag must be boolean true/false | 100% in {True, False} | **Critical** | REQ-01, REQ-02, REQ-03 |
| **DQ-PREP-01**| Prepared Layer| `track_genre`, `popularity` | Completeness & Validity | Load-ready tracks must have valid genre and popularity [0, 100] | 100% non-null & valid | **Critical** | REQ-01, REQ-02 |
| **DQ-PREP-02**| Prepared Layer| `is_grammy_winner` | Validity & Invariant | Derived winner flag must be binary integer | 100% in {0, 1} | **Critical** | REQ-02, REQ-03 |
